# LLM Compiler

### Desenvolvido por Leonardo José da Silva, seguindo a idéia do artigo https://arxiv.org/pdf/2312.04511

* Mas aplicado com uma abordagem diferente na orquestração do plano, utilizando uma estratégia de orquestração com memória

In [ ]:
from load_dotenv import load_dotenv

load_dotenv()

# CUSTOM TOOL

In [ ]:
from langchain.tools import tool
from random import randint

@tool(
    name_or_callable="cofrinhos_extrato",
    description="ferramenta para extrair extrato dos Cofrinhos, para verificar se existe fundos disponíveis."
)
def cofrinhos_extrato() -> str:
    saldo = randint(0, 10000)
    return f"Encontrado R$ {saldo},00 nos Cofrinhos do cliente"

@tool(
    name_or_callable="cofrinhos_recuperar",
    description="ferramenta para recuperar fundos dos Cofrinhos."
)
def cofrinhos_recuperar(valor: str) -> str:
    if valor is None:
        return "Qual valor deseja recuperar dos Cofrinhos?"
    return f"Recuperado {valor} dos Cofrinhos do cliente"

@tool(
    name_or_callable="execute_pix",
    description="ferramenta para executar um PIX para qualquer pessoa e qualquer valor."
)
def execute_pix(destinatario: str, valor: str) -> str:
    if destinatario is None or valor is None:
        return "Para executar um PIX, por favor forneça o destinatário e o valor juntos."
    return f"PIX de {valor} enviado para {destinatario}"

### testando as ferramentas

In [ ]:
extract = cofrinhos_extrato.invoke("")
recuperar = cofrinhos_recuperar.invoke("R$ 500,00")
pix = execute_pix.invoke({"destinatario": "João", "valor": "R$ 500,00"})

print(extract)
print(recuperar)
print(pix)


In [ ]:
tools = [cofrinhos_extrato, cofrinhos_recuperar, execute_pix]

# Planner

In [ ]:
from typing import Sequence

from langchain_core.language_models import BaseChatModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch
from langchain_core.tools import BaseTool
from langchain_core.messages import (
    BaseMessage,
    FunctionMessage,
    HumanMessage,
    SystemMessage,
)

from src.llm_compiler.output_parser import LLMCompilerPlanParser, Task
from langchain_openai import ChatOpenAI

In [ ]:
system_prompt = """Dada uma consulta do usuário, crie um plano para resolvê-la. Cada plano deve ser composto por uma ação entre os seguintes {num_tools} tipos:
{tool_descriptions}
{num_tools}. join(): Coleta e combina resultados de ações anteriores.

 - Um agente LLM é chamado ao invocar join() para finalizar a consulta do usuário ou aguardar até que os planos sejam executados.
 - join deve ser sempre a última ação do plano e será chamada em dois cenários:
  (a) se a resposta puder ser determinada reunindo as saídas das tarefas para gerar a resposta final.
  (b) se a resposta não puder ser determinada na fase de planejamento antes de executar os planos. Diretrizes:
 - Cada ação descrita acima contém tipos de entrada/saída e descrição.
   - Você deve aderir estritamente aos tipos de entrada e saída de cada ação.
   - As descrições das ações contêm diretrizes. Você DEVE seguir estritamente essas diretrizes ao usar as ações.
 - Cada ação no plano deve ser estritamente um dos tipos acima. Siga as convenções de Python para cada ação.
 - Cada ação DEVE ter um ID único, estritamente crescente.
 - As entradas das ações podem ser constantes ou saídas de ações anteriores. No segundo caso, use o formato $id para indicar o ID da ação anterior cuja saída será a entrada.
 - Sempre chame join como a última ação do plano. Escreva '<END_OF_PLAN>' depois de chamar join
 - Garanta que o plano maximize a paralelização.
 - Use apenas os tipos de ação fornecidos. Se uma consulta não puder ser atendida com eles, invoque a ação join para os próximos passos.
 - Nunca introduza novas ações além das fornecidas.

FORMATO OBRIGATÓRIO DE SAÍDA — siga EXATAMENTE este padrão de texto simples, sem JSON, sem markdown:

Thought: <raciocínio opcional>
1. nome_da_ferramenta("argumento1", "argumento2")
2. outra_ferramenta($1)
3. join()
<END_OF_PLAN>"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("placeholder", "{messages}")
])
print(prompt.pretty_print())


In [ ]:
def create_planner(
    llm: BaseChatModel, tools: Sequence[BaseTool], base_prompt: ChatPromptTemplate
):
    def _format_tool(i: int, tool) -> str:
        args_str = ", ".join(
            f"{k}: {v.get('type', 'str')}"
            for k, v in (tool.args or {}).items()
        )
        return f"{i+1}. {tool.name}({args_str}) - {tool.description}\n"

    tool_descriptions = "\n".join(
        _format_tool(i, tool)
        for i, tool in enumerate(tools)
    )
    planner_prompt = base_prompt.partial(
        replan="",
        num_tools=len(tools)
        + 1,  # Add one because we're adding the join() tool at the end.
        tool_descriptions=tool_descriptions,
    )
    replanner_prompt = base_prompt.partial(
        replan=' - Você recebe um "Plano Anterior" que é o plano criado pelo agente anterior junto com os resultados da execução '
        "(fornecidos como Observação) de cada plano e um pensamento geral (fornecido como Pensamento) sobre os resultados executados."
        'Você DEVE usar essas informações para criar o próximo plano sob "Plano Atual".\n'
        ' - Ao iniciar o Plano Atual, você deve começar com um "Pensamento" que descreve a estratégia para o próximo plano.\n'
        " - No Plano Atual, você NUNCA deve repetir as ações que já foram executadas no Plano Anterior.\n"
        " - Você deve continuar o índice de tarefas a partir do final do anterior. Não repita índices de tarefas.",
        num_tools=len(tools) + 1,
        tool_descriptions=tool_descriptions,
    )

    def should_replan(state: list):
        # Context is passed as a system message
        return isinstance(state[-1], SystemMessage)

    def wrap_messages(state: list):
        return {"messages": state}

    def wrap_and_get_last_index(state: list):
        next_task = 0
        for message in state[::-1]:
            if isinstance(message, FunctionMessage):
                next_task = message.additional_kwargs["idx"] + 1
                break
        state[-1].content = state[-1].content + f" - Begin counting at : {next_task}"
        return {"messages": state}

    return (
        RunnableBranch(
            (should_replan, wrap_and_get_last_index | replanner_prompt),
            wrap_messages | planner_prompt,
        )
        | llm
        | LLMCompilerPlanParser(tools=tools)
    )


In [ ]:
llm = ChatOpenAI(model="gpt-5.2")
planner = create_planner(llm, tools, prompt)

In [ ]:
example_question = "quero fazer um regaste dos cofrinhos de 500 reais e depois com dinheiro fazer uma transferência de pix de 100"

In [ ]:

# # Mostra o plano bruto gerado pelo LLM (antes do parsing)
# raw_output = (planner.steps[0] | planner.steps[1]).invoke([HumanMessage(content=example_question)])
# print("=" * 50)
# print("PLANO GERADO PELO LLM:")
# print("=" * 50)
# print(raw_output.content)

# # Mostra o plano parseado com ordem de execução e dependências
# print("=" * 50)
# print("ORDEM DE EXECUÇÃO:")
# print("=" * 50)
# for graph_dict in planner.stream([HumanMessage(content=example_question)]):
#     for idx, task in sorted(graph_dict.items()):
#         deps = f"depende de {task.dependencies}" if task.dependencies else "sem dependências"
#         print(f"[{idx}] {task.name}{task.args}  ({deps})")
#     print()

# print("=" * 50)
# print("EXECUÇÃO DAS TOOLS:")
# print("=" * 50)
# for graph_dict in planner.stream([HumanMessage(content=example_question)]):
#     for task in graph_dict.values():
#         tool_obj = next((t for t in tools if t.name == task.name), None)
#         if tool_obj:
#             arg_keys = list(tool_obj.args.keys())
#             args_dict = dict(zip(arg_keys, task.args))
#             print(tool_obj, args_dict)
#             result = tool_obj.invoke(args_dict)
#             print(f"Resultado: {result}")
#         else:
#             print(task.name, task.args)
#         print("---")


# Task Fetching Unit

In [ ]:
from typing import Any, Union, Iterable, List, Tuple, Dict
from typing_extensions import TypedDict
import re

from langchain_core.runnables import (
    chain as as_runnable,
)

from concurrent.futures import ThreadPoolExecutor, wait
import time


def _get_observations(messages: List[BaseMessage]) -> Dict[int, Any]:
    # Get all previous tool responses
    results = {}
    for message in messages[::-1]:
        if isinstance(message, FunctionMessage):
            results[int(message.additional_kwargs["idx"])] = message.content
    return results


class SchedulerInput(TypedDict):
    messages: List[BaseMessage]
    tasks: Iterable[Task]


def _execute_task(task, observations, config):
    tool_to_use = task.tool
    if isinstance(tool_to_use, str):
        return tool_to_use
    args = task.args
    try:
        if isinstance(args, str):
            resolved_args = _resolve_arg(args, observations)
        elif isinstance(args, dict):
            resolved_args = {
                key: _resolve_arg(val, observations) for key, val in args.items()
            }
        else:
            resolved_args = tuple(_resolve_arg(a, observations) for a in args)
    except Exception as e:
        return (
            f"ERROR(Failed to call {task.name} with args {args}.)"
            f" Args could not be resolved. Error: {repr(e)}"
        )
    try:
        if isinstance(resolved_args, dict):
            return tool_to_use(**resolved_args)
        elif isinstance(resolved_args, (list, tuple)):
            return tool_to_use(*resolved_args)
        else:
            return tool_to_use(resolved_args)
    except Exception as e:
        return (
            f"ERROR(Failed to call {task.name} with args {args}."
            + f" Args resolved to {resolved_args}. Error: {repr(e)})"
        )


def _resolve_arg(arg: Union[str, Any], observations: Dict[int, Any]):
    # $1 or ${1} -> 1
    ID_PATTERN = r"\$\{?(\d+)\}?"

    def replace_match(match):
        idx = int(match.group(1))
        return str(observations.get(idx, match.group(0)))

    # For dependencies on other tasks
    if isinstance(arg, str):
        return re.sub(ID_PATTERN, replace_match, arg)
    elif isinstance(arg, list):
        return [_resolve_arg(a, observations) for a in arg]
    else:
        return str(arg)


@as_runnable
def schedule_task(task_inputs, config):
    task: Task = task_inputs["task"]
    observations: Dict[int, Any] = task_inputs["observations"]
    try:
        observation = _execute_task(task, observations, config)
    except Exception:
        import traceback

        observation = traceback.format_exception()  # repr(e) +
    observations[task.idx] = observation


def schedule_pending_task(
    task: Task, observations: Dict[int, Any], retry_after: float = 0.2
):
    while True:
        deps = task.dependencies
        if deps and (any([dep not in observations for dep in deps])):
            # Dependencies not yet satisfied
            time.sleep(retry_after)
            continue
        schedule_task.invoke({"task": task, "observations": observations})
        break


@as_runnable
def schedule_tasks(scheduler_input: SchedulerInput) -> List[FunctionMessage]:
    """Group the tasks into a DAG schedule."""
    tasks = scheduler_input["tasks"]
    args_for_tasks = {}
    messages = scheduler_input["messages"]
    observations = _get_observations(messages)
    task_names = {}
    originals = set(observations)
    futures = []
    retry_after = 0.25  # Retry every quarter second
    with ThreadPoolExecutor() as executor:
        for task in tasks:
            deps = task.dependencies
            task_names[task.idx] = task.name
            args_for_tasks[task.idx] = task.args
            if (
                # Depends on other tasks
                deps
                and (any([dep not in observations for dep in deps]))
            ):
                futures.append(
                    executor.submit(
                        schedule_pending_task, task, observations, retry_after
                    )
                )
            else:
                # No deps or all deps satisfied
                schedule_task.invoke(dict(task=task, observations=observations))

        # All tasks have been submitted or enqueued
        # Wait for them to complete
        wait(futures)
    # Convert observations to new tool messages to add to the state
    new_observations = {
        k: (task_names[k], args_for_tasks[k], observations[k])
        for k in sorted(observations.keys() - originals)
    }
    tool_messages = [
        FunctionMessage(
            name=name, content=str(obs), additional_kwargs={"idx": k, "args": task_args}
        )
        for k, (name, task_args, obs) in new_observations.items()
    ]
    return tool_messages


In [ ]:
import itertools


@as_runnable
def plan_and_schedule(messages: List[BaseMessage], config):
    # Recupera o session_id do config para integração com o Orquestrador do Plano
    session_id = (config or {}).get("configurable", {}).get("session_id")

    tasks = planner.stream(messages, config)
    # Begin executing the planner immediately
    try:
        first = next(tasks)
    except StopIteration:
        flat_tasks = iter([])
    else:
        # planner yields graph_dict items ({idx: Task}); flatten into individual Tasks
        # e registra no Orquestrador do Plano as tarefas pendentes antes do dispatch
        def flatten_and_record(first_chunk, rest):
            if session_id:
                orch = PlanOrchestrator(redis_client, session_id)
                orch.record_pending_tasks(first_chunk)
            yield from first_chunk.values()
            for graph_dict in rest:
                if session_id:
                    orch.record_pending_tasks(graph_dict)
                yield from graph_dict.values()

        flat_tasks = flatten_and_record(first, tasks)

    scheduled_tasks = schedule_tasks.invoke(
        {
            "messages": messages,
            "tasks": flat_tasks,
        },
        config,
    )

    # Registra as tarefas concluídas no Orquestrador do Plano
    if session_id and scheduled_tasks:
        orch = PlanOrchestrator(redis_client, session_id)
        orch.record_completed_tasks(scheduled_tasks)

    return scheduled_tasks


In [ ]:
tool_messages = plan_and_schedule.invoke([HumanMessage(content=example_question)])
tool_messages

# Joiner

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage


class FinalResponse(BaseModel):
    """The final response/answer."""

    response: str


class Replan(BaseModel):
    feedback: str = Field(
        description="Analysis of the previous attempts and recommendations on what needs to be fixed."
    )


class Clarification(BaseModel):
    """Usado quando o agente precisa de informação do usuário para continuar."""

    question: str = Field(
        description=(
            "Pergunta clara e objetiva para o usuário, solicitando a informação necessária "
            "para que a execução do plano possa continuar ou ser concluída corretamente."
        )
    )


class JoinOutputs(BaseModel):
    """Decide whether to replan, ask the user for clarification, or return the final response."""

    thought: str = Field(
        description="The chain of thought reasoning for the selected action"
    )
    action: Union[FinalResponse, Replan, Clarification]


In [ ]:
joiner_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Solve a question answering task. Here are some guidelines:

 - In the Assistant Scratchpad, you will be given results of a plan you have executed to answer the user's question.

 - Thought needs to reason about the question based on the Observations in 1-2 sentences.

 - Ignore irrelevant action results.

 - If the required information is present, give a concise but complete and helpful answer to the user's question.
 - If you are unable to give a satisfactory finishing answer, replan to get the required information. Respond in the following format:
Thought: <reason about the task results and whether you have sufficient information to answer the question>
Action: <action to take>

Available actions:

 (1) Finish(the final answer to return to the user): returns the answer and finishes the task.
 (2) Replan(the reasoning and other information that will help you plan again. Can be a line of any length): instructs why we must replan
 (3) Clarify(a clear and objective question for the user): use ONLY when critical information is missing and cannot be inferred. """,
    ),
    ("placeholder", "{messages}"),
    (
        "system",
        "Using the above previous actions, decide whether to replan, clarify with the user, or finish. "
        "If all the required information is present. You may finish. "
        "If critical information is missing and the user must provide it, use Clarify. "
        "If you have made many attempts to find the information without success, "
        "admit so and respond with whatever information you have gathered so the user can work well with you.",
    ),
])


In [ ]:

runnable = joiner_prompt | llm.with_structured_output(JoinOutputs)


In [ ]:
def _parse_joiner_output(decision: JoinOutputs) -> List[BaseMessage]:
    response = [AIMessage(content=f"Thought: {decision.thought}")]
    if isinstance(decision.action, Replan):
        return response + [
            SystemMessage(
                content=f"Context from last attempt: {decision.action.feedback}"
            )
        ]
    elif isinstance(decision.action, Clarification):
        # Sinaliza ao Orquestrador do Plano que é necessário pausar para desambiguação.
        # O nó orchestrator_save detecta esta marcação e salva o estado no Redis.
        return response + [
            AIMessage(
                content=decision.action.question,
                additional_kwargs={"type": "clarification"},
            )
        ]
    else:
        return response + [AIMessage(content=decision.action.response)]


def select_recent_messages(messages: list) -> dict:
    selected = []
    observations = []

    for msg in messages[::-1]:
        if isinstance(msg, FunctionMessage):
            observations.append(msg)
        else:
            selected.append(msg)
            if isinstance(msg, HumanMessage):
                break

    selected = selected[::-1]  # restore original order

    if observations:
        observations = list(reversed(observations))  # restore original order
        obs_lines = ["Observations from executed plan:"]
        for obs in observations:
            idx = obs.additional_kwargs.get("idx", "?")
            obs_lines.append(f"  [{idx}] {obs.name}: {obs.content}")
        selected.append(HumanMessage(content="\n".join(obs_lines)))

    return {"messages": selected}


joiner = select_recent_messages | runnable | _parse_joiner_output


In [ ]:
def _parse_joiner_output(decision: JoinOutputs) -> List[BaseMessage]:
    response = [AIMessage(content=f"Thought: {decision.thought}")]
    if isinstance(decision.action, Replan):
        return response + [
            SystemMessage(
                content=f"Context from last attempt: {decision.action.feedback}"
            )
        ]
    elif isinstance(decision.action, Clarification):
        # Sinaliza ao Orquestrador do Plano que é necessário pausar para desambiguação.
        # O nó orchestrator_save detecta esta marcação e salva o estado no Redis.
        return response + [
            AIMessage(
                content=decision.action.question,
                additional_kwargs={"type": "clarification"},
            )
        ]
    else:
        return response + [AIMessage(content=decision.action.response)]


def select_recent_messages(messages: list) -> dict:
    selected = []
    observations = []

    for msg in messages[::-1]:
        if isinstance(msg, FunctionMessage):
            observations.append(msg)
        else:
            selected.append(msg)
            if isinstance(msg, HumanMessage):
                break

    selected = selected[::-1]  # restore original order

    if observations:
        observations = list(reversed(observations))  # restore original order
        obs_lines = ["Observations from executed plan:"]
        for obs in observations:
            idx = obs.additional_kwargs.get("idx", "?")
            obs_lines.append(f"  [{idx}] {obs.name}: {obs.content}")
        selected.append(HumanMessage(content="\n".join(obs_lines)))

    return {"messages": selected}


joiner = select_recent_messages | runnable | _parse_joiner_output


In [ ]:
input_messages = [HumanMessage(content=example_question)] + tool_messages

In [ ]:
# joiner.invoke(input_messages)

# Compose using LangGraph

# Orquestrador do Plano

Componente de estado global acoplado à **Task Fetching Unit** e ao **Replanner**.

Persiste no Redis (porta 6380) o estado completo de cada sessão:
- Tarefas já concluídas
- Tarefas pendentes / aguardando dependências
- Ordem e dependências entre tarefas
- Histórico de mensagens

Quando um agente emite `Clarify(...)`, a execução é **pausada** e o estado completo é salvo no Redis.
Ao receber a resposta do usuário, o sistema **retoma do ponto exato onde parou**, sem re-planejar desnecessariamente.


In [ ]:
import logging
import redis as redis_lib
from src.llm_compiler.plan_orchestrator import (
    PlanOrchestrator,
    SessionStatus,
    serialize_messages,
    deserialize_messages,
)

# Configura o logging
logging.basicConfig(level=logging.INFO)

# Conexão com o Redis local (porta 6380)
redis_client = redis_lib.Redis(
    host="localhost",
    port=6380,
    db=0,
    decode_responses=True,
    socket_connect_timeout=2,
)

# Verifica a conexão
try:
    redis_client.ping()
    logging.info("Redis conectado com sucesso na porta 6380")
except Exception as e:
    logging.error(f"Erro ao conectar ao Redis: {e}")


In [ ]:
# No do LangGraph: salva o estado no Redis quando uma desambiguacao e requisitada
@as_runnable
def orchestrator_save(messages: List[BaseMessage], config):
    """
    No do Orquestrador do Plano no LangGraph.
    Ativado quando o joiner emite uma acao Clarify.
    Salva o estado completo da execucao no Redis e encerra o ciclo atual.
    """
    session_id = (config or {}).get("configurable", {}).get("session_id")
    if not session_id:
        return []

    # Localiza a pergunta de desambiguacao no estado de mensagens
    question = ""
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.additional_kwargs.get("type") == "clarification":
            question = msg.content
            break

    if question:
        orch = PlanOrchestrator(redis_client, session_id)
        # Salva tudo exceto a mensagem de clarificacao (sera recriada ao retomar)
        messages_to_save = [
            m for m in messages
            if not (isinstance(m, AIMessage) and m.additional_kwargs.get("type") == "clarification")
        ]
        orch.pause_for_disambiguation(question, messages_to_save)
        print(f"\nDesambiguacao necessaria (session_id={session_id!r}):")
        print(f"  Pergunta: {question}")
        print(f"  Use resume_session('{session_id}', <resposta>) para continuar.\n")

    return []


In [ ]:
from langgraph.graph import MessageGraph, END
from typing import Dict

graph_builder = MessageGraph()

# 1. Define vertices
graph_builder.add_node("plan_and_schedule", plan_and_schedule)
graph_builder.add_node("join", joiner)
graph_builder.add_node("orchestrator_save", orchestrator_save)

# 2. Define edges
graph_builder.add_edge("plan_and_schedule", "join")
graph_builder.add_edge("orchestrator_save", END)


def should_continue(state: List[BaseMessage]):
    """Roteador: decide se finaliza, replana ou pausa para desambiguacao."""
    last = state[-1]
    if isinstance(last, AIMessage):
        if last.additional_kwargs.get("type") == "clarification":
            return "orchestrator_save"
        return END
    return "plan_and_schedule"


graph_builder.add_conditional_edges(
    "join",
    should_continue,
    {
        END: END,
        "plan_and_schedule": "plan_and_schedule",
        "orchestrator_save": "orchestrator_save",
    },
)
graph_builder.set_entry_point("plan_and_schedule")
chain = graph_builder.compile()


## Gerenciamento de Sessao

`run_session` — inicia uma nova execucao com rastreamento via Orquestrador do Plano.

`resume_session` — retoma uma execucao pausada, a partir do estado salvo no Redis.

`show_session_state` — inspeciona o que foi executado ate o momento em uma sessao.


In [ ]:
import json as _json
from typing import Optional


def run_session(user_query: str, session_id: Optional[str] = None):
    """
    Executa o LLMCompiler com rastreamento pelo Orquestrador do Plano.
    """
    orch = PlanOrchestrator(redis_client, session_id)
    orch.initialize_session(user_query)

    config = {"configurable": {"session_id": orch.session_id}}
    result = chain.invoke([HumanMessage(content=user_query)], config=config)
    return result, orch.session_id


def resume_session(session_id: str, user_answer: str):
    """
    Retoma uma sessao pausada para desambiguacao.

    Reconstrói o contexto sem FunctionMessages (role 'function' não é suportado
    por modelos mais recentes). As observações das tarefas já concluídas são
    incorporadas diretamente no SystemMessage de contexto para o Replanner.
    """
    orch = PlanOrchestrator(redis_client, session_id)
    saved_state = orch.resume(user_answer)

    if saved_state is None:
        raise ValueError(
            f"Sessao '{session_id}' nao encontrada ou nao esta pausada para desambiguacao.\n"
            "Verifique o session_id ou se o Redis esta acessivel."
        )

    # Monta o resumo das tarefas já concluídas como texto (evita FunctionMessage)
    completed = saved_state.get("completed_tasks", {})
    obs_lines = []
    if completed:
        obs_lines.append("Tarefas ja concluidas na execucao anterior:")
        for idx in sorted(completed, key=lambda x: int(x)):
            t = completed[idx]
            obs_lines.append(f"  [{idx}] {t['name']} -> {t['observation']}")

    context = (
        f"O usuario respondeu a pergunta de desambiguacao com: '{user_answer}'. "
        "Continue a execucao a partir das tarefas ja concluidas, sem repeti-las. "
        "Se a resposta fornecida for suficiente para concluir o objetivo original, finalize diretamente."
    )
    if obs_lines:
        context += "\n\n" + "\n".join(obs_lines)

    # Reconstrói as mensagens sem FunctionMessage para compatibilidade com a API
    original_query = saved_state.get("original_query", "")
    messages = [
        HumanMessage(content=original_query),
        HumanMessage(content=user_answer),
        SystemMessage(content=context),  # SystemMessage aciona o caminho do Replanner
    ]

    config = {"configurable": {"session_id": session_id}}
    return chain.invoke(messages, config=config)


def show_session_state(session_id: str):
    """Exibe o estado atual da sessao no Orquestrador do Plano."""
    orch = PlanOrchestrator(redis_client, session_id)
    summary = orch.get_summary()
    print(_json.dumps(summary, indent=2, ensure_ascii=False, default=str))


## Exemplo de Uso — Desambiguacao e Retomada

```python
# 1. Inicia a execucao com rastreamento
result, session_id = run_session("quero fazer um resgate e pix")

# Se o agente precisou de desambiguacao, a execucao fica pausada.
# O session_id e a chave para retomar em qualquer momento:
show_session_state(session_id)

# 2. Usuario responde e a execucao retoma do ponto exato
result = resume_session(session_id, "R$ 300,00 do cofrinho principal")
print(result[-1].content)
```


In [ ]:
from IPython.display import Image, display

display(Image(chain.get_graph().draw_mermaid_png()))

In [ ]:
for step in chain.stream([HumanMessage(content=example_question)]):
    print(step)
    print("---")

In [ ]:
# Final answer
last_messages = next(iter(step.values()))
print(last_messages[-1].content)


In [ ]:

# ── 1. RODAR SESSÃO ──────────────────────────────────────────────────────────
# Use uma pergunta propositalmente ambígua para forçar desambiguação.
active_session_id = None

result_or_pause, active_session_id = run_session(
    "quero fazer um resgate e depois um pix"
)

# Verifica se a sessão foi pausada para desambiguação
_state = PlanOrchestrator(redis_client, active_session_id).load_state()
session_paused = _state and _state.get("status") == SessionStatus.PAUSED_FOR_DISAMBIGUATION

if session_paused:
    print("=" * 60)
    print("⚠️  SESSÃO PAUSADA — DESAMBIGUAÇÃO NECESSÁRIA")
    print("=" * 60)
    print(f"  session_id : {active_session_id}")
    print(f"  Pergunta   : {_state.get('disambiguation_question')}")
    print()
    print("→ Execute a célula do LOG e depois a célula de RESPOSTA.")
else:
    print("=" * 60)
    print("✅  EXECUÇÃO CONCLUÍDA (sem desambiguação necessária)")
    print("=" * 60)
    if result_or_pause:
        print(result_or_pause[-1].content)


In [ ]:

# ── 2. LOG DO ORQUESTRADOR DO PLANO ─────────────────────────────────────────
def show_plan_log(session_id: str) -> None:
    """Exibe o log completo do estado do Orquestrador do Plano para a sessão."""
    orch = PlanOrchestrator(redis_client, session_id)
    s = orch.load_state()
    if s is None:
        print(f"[PlanOrchestrator] Sessão '{session_id}' não encontrada no Redis.")
        return

    status_icon = {
        "running": "🔄",
        "paused_for_disambiguation": "⏸️ ",
        "completed": "✅",
    }.get(s.get("status", ""), "❓")

    sep = "─" * 60
    print(sep)
    print("📋  LOG DO ORQUESTRADOR DO PLANO")
    print(sep)
    print(f"  session_id   : {s['session_id']}")
    print(f"  status       : {status_icon} {s['status']}")
    print(f"  query        : {s['original_query']}")
    print(f"  criado em    : {s.get('created_at', '-')}")
    print(f"  atualizado   : {s.get('updated_at', '-')}")

    print()
    print("  TAREFAS PENDENTES:")
    pending = s.get("pending_tasks", {})
    if pending:
        for idx in sorted(pending, key=lambda x: int(x)):
            t = pending[idx]
            deps = t.get("dependencies") or []
            dep_str = f"deps={deps}" if deps else "sem deps"
            print(f"    [{idx}] {t['name']}({', '.join(str(a) for a in t['args'])})  {dep_str}")
    else:
        print("    (nenhuma)")

    print()
    print("  TAREFAS CONCLUÍDAS:")
    completed = s.get("completed_tasks", {})
    if completed:
        for idx in sorted(completed, key=lambda x: int(x)):
            t = completed[idx]
            print(f"    [{idx}] {t['name']}  →  {t['observation']}")
    else:
        print("    (nenhuma)")

    print()
    print("  HISTÓRICO DE MENSAGENS:")
    messages = s.get("messages", [])
    if messages:
        for i, m in enumerate(messages):
            role = m.get("type", "?")
            content = str(m.get("content", ""))[:100]
            suffix = "..." if len(str(m.get("content", ""))) > 100 else ""
            print(f"    [{i}] {role}: {content}{suffix}")
    else:
        print("    (nenhuma mensagem salva)")

    disam_q = s.get("disambiguation_question")
    if disam_q:
        print()
        print("  ⏸️  PERGUNTA DE DESAMBIGUAÇÃO:")
        print(f"    {disam_q}")

    final = s.get("final_response")
    if final:
        print()
        print("  ✅ RESPOSTA FINAL:")
        print(f"    {final}")

    print(sep)


# Executa o log para a sessão ativa
if active_session_id:
    show_plan_log(active_session_id)
else:
    print("Nenhuma sessão ativa. Execute a célula anterior primeiro.")


In [ ]:

# ── 3. RESPONDER À PERGUNTA DE DESAMBIGUAÇÃO ─────────────────────────────────
# Digite sua resposta quando o input aparecer e pressione Enter.
# O agente retomará a execução do ponto exato onde parou.

if not active_session_id:
    print("Nenhuma sessão ativa. Execute a célula de 'RODAR SESSÃO' primeiro.")
else:
    _state_check = PlanOrchestrator(redis_client, active_session_id).load_state()
    if not _state_check or _state_check.get("status") != SessionStatus.PAUSED_FOR_DISAMBIGUATION:
        print("A sessão não está pausada para desambiguação.")
        print(f"Status atual: {_state_check.get('status') if _state_check else 'não encontrada'}")
    else:
        print(f"Pergunta do agente: {_state_check.get('disambiguation_question')}")
        print()
        sua_resposta = input("✏️  Sua resposta: ")

        print(f"\nRetomando sessão '{active_session_id}' com a resposta: '{sua_resposta}'\n")
        final_result = resume_session(active_session_id, sua_resposta)

        print("=" * 60)
        print("✅  RESPOSTA FINAL DO AGENTE")
        print("=" * 60)
        print(final_result[-1].content)

        print()
        print("── Log pós-retomada ──")
        show_plan_log(active_session_id)
